# Purchase-model training and pickle example

Train a simple linear regression on the supplied **purchase decisions** and save it for the pricing examples. The model estimates purchase probability from covariates and the offered price.

In [ ]:
from pathlib import Path
import os
import sys

# Resolve paths once so every notebook uses the project folder.
project_folder = next((
    folder for parent in (Path.cwd(), *Path.cwd().parents)
    for folder in (parent, parent / 'project_code_2026_publicshare',
                   parent / 'project_2026/project_code_2026_publicshare',
                   parent / 'private_github/project_2026/project_code_2026_publicshare')
    if (folder / 'pricing_helpers.py').is_file()
), None)
if project_folder is None:
    raise FileNotFoundError('Open the project folder as your notebook workspace.')
os.chdir(project_folder)
sys.path.insert(0, str(project_folder))

import pickle
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

data = pd.read_csv(project_folder / 'data/train_prices_decisions_2026.csv')


In [ ]:
features = ['Covariate1', 'Covariate2', 'Covariate3', 'price_item']
train, test = train_test_split(data, test_size=.2, random_state=0)
model = LinearRegression().fit(train[features], train.item_bought.astype(float))
# Clip regression predictions to [0, 1]. For a classifier, you may
# need predict_proba() instead of predict(), using the column for purchase = 1.
purchase_probability = np.clip(model.predict(test[features]), 0, 1)
print('Held-out mean squared error:', np.mean((purchase_probability-test.item_bought)**2))


## Save and reload the model

Commit your training code here and the saved model in `agents/helper_files_here/`. Both `agents/agent_part1.py` and `agents/agent_part2.py` load this model.


In [ ]:
model_path = project_folder / 'agents/helper_files_here/purchase_model.pkl'

#example on how to save a model in a pickle file
with model_path.open('wb') as file:
    pickle.dump(model, file)
    
#how to open the saved model -- you will need to do this in your agents file
with model_path.open('rb') as file:
    loaded_model = pickle.load(file)
    
np.testing.assert_allclose(loaded_model.predict(test[features]), model.predict(test[features]))
print('Saved and reloaded:', model_path)

In [ ]:
example = test[features].iloc[:5]
probability = np.clip(loaded_model.predict(example), 0, 1)
example.assign(predicted_purchase_probability=probability,
               predicted_revenue=example.price_item*probability)